# 🛒 ApnaMart AI Sales Associate (ApnaMart Sahayak)
### An Agentic E-Commerce RAG System Replicating In-Store Consultative Product Discovery

**Live Target Website:** [https://apnamarts.shop/](https://apnamarts.shop/) (*"Sab Kuch, Ek Hi Jagah"*)
**Primary Architectural Reference:** [TheGreatBonnie/ecommerce-rag-agent](https://github.com/TheGreatBonnie/ecommerce-rag-agent)
**Key AI Stack:**
- **LLM Reasoning & Dialogue:** Groq LPU (`llama-3.3-70b-versatile` / `gpt-oss-20b` equivalent)
- **Dense Semantic Embedding:** Hugging Face [`sentence-transformers/all-mpnet-base-v2`](https://huggingface.co/sentence-transformers/all-mpnet-base-v2) (768-Dimensional)
- **Vector Similarity & Constraint Retrieval:** Hybrid Cosine Similarity with Multi-Facet Constraint Filtering (Price, Rating, Category, Size, Color)
- **Agentic Orchestration:** LangGraph 5-node `StateGraph` workflow with real-time state logging
- **Backend / Frontend:** FastAPI REST backend + turnkey Interactive Storefront with floating AI assistant


## 1. System Architecture & Information Flow

```text
CUSTOMER / USER
       │
       ▼
┌────────────────────────────────────────────────────────────────────────┐
│ Node 1: EXTRACT INTENT & SLOTS                                       │
│ Parse Category -> Subcategory -> Type -> Occasion -> Size -> Color   │
│ Extract Constraints: Price Max/Min, Min Star Rating, In-Stock        │
└────────────────────────────────────────────────────────────────────────┘
       │
       ▼
┌────────────────────────────────────────────────────────────────────────┐
│ Node 2: SYNTHESIZE QUERY (Groq LPU - llama-3.3-70b / gpt-oss-20b)     │
│ "User wants an M-sized navy-blue or black suit for office wear."     │
└────────────────────────────────────────────────────────────────────────┘
       │
       ▼
┌────────────────────────────────────────────────────────────────────────┐
│ Node 3: DENSE EMBEDDING (sentence-transformers/all-mpnet-base-v2)     │
│ 768-D Query Vector (q_vec ∈ ℝ⁷⁶⁸)                                  │
└────────────────────────────────────────────────────────────────────────┘
       │
       ▼
┌────────────────────────────────────────────────────────────────────────┐
│ Node 4: VECTOR RETRIEVAL & HYBRID RANKING                            │
│ Cosine Sim(q, p_i) + Constraints + Category/Type Boost over 103 items │
│ Top-5 Candidate Products Selected                                     │
└────────────────────────────────────────────────────────────────────────┘
       │
       ▼
┌────────────────────────────────────────────────────────────────────────┐
│ Node 5: CONSULTATIVE SALES PITCH (ApnaMart Sahayak)                   │
│ Personalized offline retail associate pitch explaining WHY each item   │
│ matches the user's specific occasion, fit, styling, and budget        │
└────────────────────────────────────────────────────────────────────────┘
       │
       ▼
FINAL RECOMMENDATION & PRODUCT CARDS (Add to Bag / Buy Now)
```


In [ ]:
# Step 1: Environment Setup and Package Import
import os
import sys
import json
import time
import numpy as np
from pathlib import Path
from dotenv import load_dotenv

# Ensure UTF-8 output on Windows
if hasattr(sys.stdout, 'reconfigure'):
    sys.stdout.reconfigure(encoding='utf-8')

# Add parent directory to sys.path
ROOT_DIR = Path.cwd().parent
if str(ROOT_DIR) not in sys.path:
    sys.path.insert(0, str(ROOT_DIR))

load_dotenv(ROOT_DIR / '.env')

from ecommerce_agent import (
    graph,
    search_ecommerce,
    search_products,
    get_product_recommendation,
    get_embedding,
    initial_products,
    setup_mongodb,
    clear_all_caches
)

print(f"[OK] System Initialized successfully!")
print(f"[OK] Reference Repo: TheGreatBonnie/ecommerce-rag-agent")
print(f"[OK] Total Scraped & Indexed Products: {len(initial_products)}")


In [ ]:
# Step 2: Catalog Breakdown by Department
from collections import Counter

categories = Counter(p.get('category') for p in initial_products)
print("=== ApnaMart Scraped Catalog Breakdown (https://apnamarts.shop/) ===")
for cat, count in categories.most_common():
    print(f"  • {cat:<18}: {count:>3} products")

# Inspect a sample product with pre-computed 768-D MPNet embedding
sample = initial_products[0]
print("\n--- Sample Catalog Item ---")
print(f"ID: {sample.get('id')}")
print(f"Name: {sample.get('name')}")
print(f"Category: {sample.get('category')} > {sample.get('sub_category')}")
print(f"Price: ₹{sample.get('price')} (Original: ₹{sample.get('original_price')}, {sample.get('discount')} OFF)")
print(f"Rating: {sample.get('rating')}⭐ ({sample.get('reviews_count')} reviews)")
print(f"Sizes: {sample.get('sizes')}")
print(f"Colors: {sample.get('colors')}")
print(f"Embedding Dimension: {len(sample.get('embedding', []))} (sentence-transformers/all-mpnet-base-v2)")


## 2. Mathematical Formulation: 768-D Dense Embeddings & Vector Cosine Similarity

Given a synthesized customer query sentence $Q$, the sentence transformer converts it to a 768-dimensional normalized dense embedding vector:

$$\vec{q} = \text{all-mpnet-base-v2}(Q) \in \mathbb{R}^{768}, \quad \|\vec{q}\|_2 = 1$$

For each product $P_i$ with pre-computed representation vector $\vec{p}_i \in \mathbb{R}^{768}$, the semantic proximity is calculated using Cosine Similarity:

$$\text{Sim}_{\text{cos}}(\vec{q}, \vec{p}_i) = \frac{\vec{q} \cdot \vec{p}_i}{\|\vec{q}\|_2 \|\vec{p}_i\|_2} = \sum_{j=1}^{768} q_j \cdot p_{i,j}$$

To replicate an offline sales associate's precise constraint adherence, the system combines semantic similarity with categorical and slot boosts:

$$\text{Score}(P_i) = \text{Sim}_{\text{cos}}(\vec{q}, \vec{p}_i) + \beta_{\text{type}} \cdot \mathbb{I}_{\text{type}} + \beta_{\text{sub}} \cdot \mathbb{I}_{\text{sub}} + \beta_{\text{occ}} \cdot \mathbb{I}_{\text{occasion}} + \beta_{\text{size}} \cdot \mathbb{I}_{\text{size}}$$


In [ ]:
# Step 3: Test Scenario 1 — The Core Offline Sales Associate Experience (Men's Suit)
# User flow: Clothes -> Men -> Suit -> Full Suit -> Office Wear -> Size M -> Navy Blue / Black

collection = setup_mongodb()
query_1 = "I want an M-sized navy blue or black full suit for office wear and interview"

print(f"Executing vector search for query: '{query_1}'")
top_suits = search_products(collection, query_1, limit=3)

print(f"\n[Top 3 Retrieved Products via 768-D Cosine Similarity]:")
for i, p in enumerate(top_suits, 1):
    print(f"{i}. {p['name']}")
    print(f"   Price: ₹{p['price']:,} | Rating: {p['rating']}⭐ | Similarity: {p['similarity_score']}")
    print(f"   Sizes: {', '.join(p['sizes'])} | Colors: {', '.join(p['colors'])}")

# Generate the consultative pitch
pitch_1 = get_product_recommendation(top_suits, query_1)
print("\n--- Consultative Sales Associate Pitch (ApnaMart Sahayak) ---")
print(pitch_1)


In [ ]:
# Step 4: Test Scenario 2 — Constraint Search: 5G Smartphone under ₹20,000
query_2 = "5G smartphone under 20000 with 128GB or 256GB for daily use"

print(f"Executing vector search for: '{query_2}'")
top_phones = search_products(collection, query_2, limit=3)

print(f"\n[Top 3 Phones Retrieved Under Budget]:")
for i, p in enumerate(top_phones, 1):
    print(f"{i}. {p['name']}")
    print(f"   Price: ₹{p['price']:,} (Under ₹20,000 Verified) | Score: {p['similarity_score']}")


In [ ]:
# Step 5: Test Scenario 3 — Beauty & Skincare
query_3 = "Derma sunscreen SPF 50 or vitamin C serum for daily skincare"

top_beauty = search_products(collection, query_3, limit=3)
print(f"\n[Top 3 Beauty & Skincare Matches]:")
for i, p in enumerate(top_beauty, 1):
    print(f"{i}. {p['name']} | Price: ₹{p['price']:,} | Rating: {p['rating']}⭐")


In [ ]:
# Step 6: Full LangGraph 5-Node StateGraph Execution & Trace Inspection
import asyncio

async def run_langgraph_test():
    state_input = {
        "current_query": "I need a navy blue suit in size M for an office interview",
        "category": "Fashion",
        "sub_category": "Men",
        "attributes": {"size": "M", "color": "Navy Blue", "occasion": "Interview", "type": "Suit"},
        "logs": []
    }
    
    print("Invoking LangGraph StateGraph...")
    result = await graph.ainvoke(state_input)
    
    print("\n=== LangGraph State Execution Trace ===")
    for log in result.get('logs', []):
        print(f"  ✓ {log['message']}")
        
    print("\nSynthesized Query Sentence:", result.get('query_sentence'))
    print("Top Matched Products Count:", len(result.get('top_products', [])))
    print("Final Stage Status:", result.get('search_stage'))

await run_langgraph_test()


## 3. Comparison with Reference Repository (`TheGreatBonnie/ecommerce-rag-agent`)

| Architectural Feature | TheGreatBonnie/ecommerce-rag-agent | ApnaMart AI Sales Associate (This System) |
| :--- | :--- | :--- |
| **Domain & Catalog** | 60 synthetic tech items | 103 verified items scraped live from [https://apnamarts.shop/](https://apnamarts.shop/) |
| **Retail Persona** | Generic chat assistant | Offline Sales Associate ("ApnaMart Sahayak") replicating in-store consultative dialogue |
| **Product Discovery** | Free-form search bar only | Dual-mode: Cascading 8-department Dropdown Wizard + Conversational AI Chat |
| **LLM Inference** | OpenAI API | Groq LPUs (`llama-3.3-70b-versatile` / `gpt-oss-20b` equivalent) for sub-second latency |
| **Embedding Architecture** | OpenAI 1536-D | Hugging Face `sentence-transformers/all-mpnet-base-v2` 768-D dense vectors |
| **Constraint Handling** | Simple text splits | Comprehensive natural-language regex + hybrid BM25 metadata scoring |
| **Workflow Engine** | LangGraph StateGraph | LangGraph 5-node StateGraph (`extract_intent` → `synthesize_query` → `embed_query` → `retrieve_and_rank` → `generate_pitch`) |
| **Turnkey Execution** | Requires MongoDB Atlas cluster | Turnkey In-Memory Vector Collection fallback + optional MongoDB Atlas |
| **Web UI Experience** | Basic CopilotKit frame | Full ApnaMart storefront backdrop + interactive floating widget with deals & cart |
